1. [Marks: 40%] Combine three of the 25 most downloaded books from
Project Gutenberg
(https://www.gutenberg.org/ebooks/search/?sort_order=downloads) to
train an English text generator. You are required to use an RNN-based
architecture only, such as Simple RNN, LSTM, or GRU. Pretrained
transformer models and other large pretrained language models, including
BERT, GPT, RoBERTa, DistilBERT, T5, or similar models, are not
permitted. Submit your answer to this question in a Jupyter notebook file
called books.ipynb.
2. [Marks: 60%] Then, use your model from Question 1 above to generate a
text consisting of approx. 100 words. Submit your answer to this question
in the same Jupyter notebook of Question 1, and include in the notebook
the 100-word text generated by your code.


##
A character-level LSTM architecture was chosen to demonstrate the model's ability to learn English syntax, morphology, and punctuation from scratch.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout

# 3 Project Gutenberg top 25 .txt files converted them to all lower case
file_paths = ['Crime.txt', 'Frankenstein.txt', 'Gatsby.txt']
text = ""
for path in file_paths:
    with open(path, 'r', encoding='utf-8') as f:
        text += f.read().lower()

# Create character mappings
chars = sorted(list(set(text)))
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}

n_chars = len(text)
n_vocab = len(chars)
print(f"Total Characters: {n_chars}")
print(f"Unique Characters (Vocab Size): {n_vocab}")

# Prepare the dataset (sliding window)
seq_length = 100 # Number of characters to look at to predict the next
dataX = []
dataY = []

# To save time/memory, we step by 15 instead of 1
for i in range(0, n_chars - seq_length, 15):
    seq_in = text[i:i + seq_length]
    seq_out = text[i + seq_length]
    dataX.append([char_to_int[char] for char in seq_in])
    dataY.append(char_to_int[seq_out])


n_patterns = len(dataX)
X = np.reshape(dataX, (n_patterns, seq_length, 1)) / float(n_vocab)
y = tf.keras.utils.to_categorical(dataY)

Total Characters: 1883383
Unique Characters (Vocab Size): 78


In [2]:
model = Sequential([
    # LSTM Layer 1
    LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
    Dropout(0.2),
    # LSTM Layer 2
    LSTM(256),
    Dropout(0.2),
    # Output Layer
    Dense(y.shape[1], activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

/opt/anaconda3/envs/tf/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 256)       │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 77)             │        19,789 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 809,293 (3.09 MB)

 Trainable params: 809,293 (3.09 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Training 
model.fit(X, y, epochs=20, batch_size=128)
# Model save provided by Gemini AI in order to not have to run model again
model.save('text_gen.keras')





Epoch 1/20
 15/981 ━━━━━━━━━━━━━━━━━━━━ 4:45 296ms/step - loss: 2.0876

KeyboardInterrupt: 

In [28]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Mapping for model loading must be the same as above
file_paths = ['Crime.txt', 'Frankenstein.txt', 'Gatsby.txt']
text = ""
for path in file_paths:
    with open(path, 'r', encoding='utf-8') as f:
        text += f.read().lower()

chars = sorted(list(set(text)))
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}
n_vocab = len(chars)
seq_length = 100 

# load the trained model
model = load_model('text_gen.keras')
print("Model loaded successfully!")

# DEFINE GENERATION FUNCTION
def generate_final_output(seed_text, words_to_generate=100, temperature=0.6):
    generated = seed_text
    words_count = 0
    
    for _ in range(2000):
        if words_count >= words_to_generate:
            break
            
        # Grab the last 100 characters for the prediction window
        input_seq = generated[-seq_length:]
        x_pred = np.reshape([char_to_int[c] for c in input_seq], (1, seq_length, 1))
        x_pred = x_pred / float(n_vocab)
        
        preds = model.predict(x_pred, verbose=0)[0]
        
        # Make sure predictions match the character dictionary size
        if len(preds) > len(chars):
            preds = preds[:len(chars)]  # Truncate if needed
        
        # Normalize the predictions to sum to 1
        preds = preds / np.sum(preds) 
        
        # Apply temperature for better flow
        preds = np.asarray(preds).astype('float64')
        preds = np.log(preds + 1e-7) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)
        
        # Make sure the choices array and probability array have the same length
        choices = range(len(preds))  # Use length of preds instead of chars
        next_index = np.random.choice(choices, p=preds)
        
        # Make sure the index is valid for int_to_char
        if next_index < len(chars):
            next_char = int_to_char[next_index]
            generated += next_char
            if next_char == " ":
                words_count += 1
        
    return generated


# Use a seed that is at least 100 characters long 
# asked Chatgpt for a list of 100 character seeds based on the 3 books and choose 1
raw_seed = "the wind had blown off, leaving a loud, bright night, with wings beating in the trees and a persistent"
prepared_seed = raw_seed.lower().rjust(100)

final_text = generate_final_output(prepared_seed, words_to_generate=100)

print("GENERATED 100-WORD TEXT")
print(final_text)

Model loaded successfully!
GENERATED 100-WORD TEXT
the wind had blown off, leaving a loud, bright night, with wings beating in the trees and a persistent arynh aoient toiet bnrvh sornd to
the oefefet. the sas, but the dacr ofcsly io the coon was sried hi the calllt
of hose no the maxes anungrt orotes hn a dound fane of an ofcslee of a pearenee gole fir tore mnoe fatpeu, “i was so have eound the wouk in the woolg of mot. le you are srrp to teie wour mane corn the oevtra wou ane so that ael ho the aouiend of this oeict. codatene to de aoemp tp le. ie lay pome tal iang of ht frmm thi aavtt li the tonn of tascsier and that 


## GENERATED 100-WORD TEXT
the wind had blown off, leaving a loud, bright night, with wings beating in the trees and a persistent arynh aoient toiet bnrvh sornd to
the oefefet. the sas, but the dacr ofcsly io the coon was sried hi the calllt
of hose no the maxes anungrt orotes hn a dound fane of an ofcslee of a pearenee gole fir tore mnoe fatpeu, “i was so have eound the wouk in the woolg of mot. le you are srrp to teie wour mane corn the oevtra wou ane so that ael ho the aouiend of this oeict. codatene to de aoemp tp le. ie lay pome tal iang of ht frmm thi aavtt li the tonn of tascsier and that